# ⚡ Módulo 07: Sistemas, Inferencia Eficiente y Cuantización
## Capítulo 3: Atención con Conciencia de Hardware: De la Atención Estándar $O(N^2)$ a FlashAttention (Tiling, SRAM vs HBM y Recomputing)

> *"El cuello de botella de la atención en Transformers nunca fue el número de operaciones matemáticas (FLOPs), sino la transferencia de datos a través de la memoria. En la atención estándar, materializar la matriz de atención $S = Q K^T$ de tamaño $N \times N$ en la memoria VRAM global (HBM) requiere $O(N^2)$ escrituras y lecturas lentas, provocando errores de Out-Of-Memory en secuencias largas. En 2022, Tri Dao y el equipo de Stanford rompieron este límite con FlashAttention: demostraron que calculando la atención por bloques en la memoria ultrarrápida del chip (SRAM) mediante el truco del Online Softmax y recalculando los bloques intermedios en el backward pass, es posible ejecutar la atención exacta con $O(N)$ accesos a memoria y una aceleración de $2\times$ a $4\times$."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/07_systems_and_efficiency/03_hardware_aware_attention.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías matemáticas y fijamos semillas para asegurar reproducibilidad determinista.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict, Optional
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para estudiar FlashAttention, Online Softmax y Tiling en bloques")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### La Jerarquía Física de Memoria en una GPU
Una GPU moderna (o una APU con memoria compartida) posee una jerarquía estricta de memorias:
1. **HBM (High Bandwidth Memory) / DRAM (Memoria Global):**
   * Capacidad: Grande (16 GB - 80 GB).
   * Ancho de banda: Moderado (en una A100 es de $\approx 2.0$ TB/s; en DDR5 de consumo es de $\approx 80$ GB/s).
   * Latencia: Alta (cientos de ciclos de reloj).
2. **SRAM (Shared Memory / L1 Cache en el chip):**
   * Capacidad: Minúscula (apenas $\approx 100 - 256$ KB por Streaming Multiprocessor).
   * Ancho de banda: **Monstruoso ($\approx 19$ TB/s, unas $10\times$ más rápido que HBM)**.
   * Latencia: Cero (1 ciclo de reloj).

### La Pesadilla de Memoria de la Atención Estándar (Vaswani et al., 2017)
La ecuación canónica de atención es:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

En la implementación tradicional de PyTorch:
1. Carga $Q$ y $K$ de HBM a SRAM $\implies$ Multiplica $S = Q K^T$ $\implies$ **Escribe la matriz $N \times N$ en HBM**.
2. Lee la matriz $S$ desde HBM a SRAM $\implies$ Calcula $P = \text{softmax}(S)$ $\implies$ **Escribe la matriz $P$ ($N \times N$) en HBM**.
3. Lee $P$ y $V$ desde HBM a SRAM $\implies$ Multiplica $O = P V$ $\implies$ Escribe el resultado $O$ ($N \times d$) en HBM.

* **El Desastre:** Para una secuencia de $N = 16.384$ tokens, la matriz $N \times N$ en FP16 ocupa **536 MB por cada cabeza de atención**. Para 32 cabezas, ¡son **17 GB de memoria únicamente para guardar la matriz de atención intermedia**!
* La GPU pasa el $80\%$ de su tiempo bloqueada transfiriendo matrices cuadráticas por el bus de memoria HBM, cayendo en el **Muro de la Memoria**.

### El Momento Eureka: FlashAttention (Tri Dao et al., Stanford 2022)
Tri Dao y sus colaboradores de Stanford se hicieron una pregunta revolucionaria:
> *"¿Podemos calcular la atención EXACTA sin escribir jamás la matriz $N \times N$ en la memoria global HBM?"*

La respuesta fue afirmativa gracias a dos ideas complementarias:
1. **Tiling (Enlosado en Bloques):** Dividir las matrices $Q, K, V$ en bloques que quepan holgadamente en la memoria SRAM ultrarrápida del procesador.
2. **Online Softmax:** Una reformulación algebraica que permite calcular el Softmax de forma acumulativa y exacta bloque a bloque sin necesitar ver toda la secuencia a la vez.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### El Desafío Matemático del Softmax Clásico
Dado un vector $x \in \mathbb{R}^N$, el Softmax estable requiere dos pasadas globales:
1. Hallar el máximo global: $m = \max_i x_i$.
2. Calcular la suma de exponenciales: $d = \sum_i e^{x_i - m}$.
3. Normalizar: $p_i = \frac{e^{x_i - m}}{d}$.

¡Parece imposible calcular el Softmax por bloques porque necesitas el máximo global $m$ antes de poder sumar los exponenciales!

### La Solución Elegante: Online Softmax (Milakov & Gimelshtein, 2018)
Supongamos que ya procesamos un bloque $A$ y conocemos su máximo local $m_A$ y su suma exponencial $d_A$.
Ahora recibimos un nuevo bloque $B$ con su máximo local $m_B$ y su suma $d_B$:

1. **Nuevo máximo global combinado:**
   $$\mathbf{m_{new} = \max(m_A, m_B)}$$

2. **Actualización de la suma normalizadora sin recalcular el bloque $A$:**
   $$\mathbf{d_{new} = d_A \cdot e^{m_A - m_{new}} + d_B \cdot e^{m_B - m_{new}}}$$

3. **Actualización del acumulador de salida $O$:**
   $$\mathbf{O_{new} = O_A \cdot \left( \frac{d_A \cdot e^{m_A - m_{new}}}{d_{new}} \right) + (P_B V_B) \cdot \left( \frac{e^{m_B - m_{new}}}{d_{new}} \right)}$$

¡Cada bloque ajusta recursivamente la escala de la salida previa mediante el factor de corrección $e^{m_A - m_{new}}$!
Al terminar todos los bloques, **el resultado es idéntico bit a bit a la atención global**, pero habiendo usado únicamente memoria $O(N)$ en lugar de $O(N^2)$.

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

### Paso 1: Verificación de Online Softmax en 1D
Comprobemos que procesar un vector en bloques con Online Softmax produce exactamente el mismo resultado que `torch.softmax` global:

In [ ]:
def online_softmax(x: torch.Tensor, block_size: int = 4) -> torch.Tensor:
    """
    Calcula Softmax de forma streaming bloque a bloque sin materializar todo a la vez.
    """
    N = x.shape[0]
    m_prev = float('-inf')
    d_prev = 0.0
    
    # En el algoritmo streaming acumulamos los numeradores reescalados
    out = torch.zeros_like(x)
    
    for i in range(0, N, block_size):
        x_block = x[i:i+block_size]
        m_block = float(torch.max(x_block))
        
        # Nuevo maximo combinado
        m_curr = max(m_prev, m_block)
        
        # Re-escalar suma previa y sumar nuevo bloque
        d_curr = d_prev * math.exp(m_prev - m_curr) + float(torch.sum(torch.exp(x_block - m_curr)))
        
        # Re-escalar las salidas ya calculadas y calcular las nuevas
        if i > 0:
            out[:i] = out[:i] * (d_prev * math.exp(m_prev - m_curr) / d_curr)
            
        p_block = torch.exp(x_block - m_curr) / d_curr
        out[i:i+block_size] = p_block
        
        m_prev = m_curr
        d_prev = d_curr
        
    return out

# Prueba de identidad numérica
x_test = torch.tensor([1.2, 5.4, 3.1, -0.5, 8.2, 2.0, 4.4, -1.1])
expected_sm = F.softmax(x_test, dim=0)
online_sm = online_softmax(x_test, block_size=3)

print("Softmax Estándar:", expected_sm.tolist()[:4])
print("Online Softmax:  ", online_sm.tolist()[:4])
assert torch.allclose(expected_sm, online_sm, atol=1e-6)
print("✅ Online Softmax coincide exactamente con el Softmax clásico sin ver el vector completo")

### Paso 2: FlashAttention Forward Pass en Bloques From Scratch
Implementemos el algoritmo de FlashAttention dividiendo $Q$ en bloques $B_r$ y $K, V$ en bloques $B_c$:

In [ ]:
def flash_attention_forward(
    Q: torch.Tensor, 
    K: torch.Tensor, 
    V: torch.Tensor, 
    block_size_q: int = 16, 
    block_size_kv: int = 16
) -> torch.Tensor:
    """
    Algoritmo FlashAttention (Tri Dao et al., 2022) implementado desde primeros principios.
    
    Q, K, V: Matrices de tamano (N, d)
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    
    # Matriz de salida y acumuladores estadisticos en memoria rapida
    O = torch.zeros_like(Q)
    l = torch.zeros(N)                     # Suma de denominadores (normalizadores)
    m = torch.full((N,), float('-inf'))   # Maximos locales para estabilidad
    
    # Bucle externo: recorre bloques de Query (Br)
    for i in range(0, N, block_size_q):
        q_block = Q[i:i+block_size_q]  # (Br, d)
        o_block = O[i:i+block_size_q]  # (Br, d)
        l_block = l[i:i+block_size_q]  # (Br,)
        m_block = m[i:i+block_size_q]  # (Br,)
        
        # Bucle interno: recorre bloques de Key y Value (Bc)
        for j in range(0, N, block_size_kv):
            k_block = K[j:j+block_size_kv]  # (Bc, d)
            v_block = V[j:j+block_size_kv]  # (Bc, d)
            
            # 1. Producto local en SRAM: S_ij = Q_i @ K_j^T * scale
            S_ij = (q_block @ k_block.t()) * scale  # (Br, Bc)
            
            # 2. Maximo local del nuevo bloque
            m_ij, _ = torch.max(S_ij, dim=-1)  # (Br,)
            
            # 3. Nuevo maximo combinado
            m_new = torch.maximum(m_block, m_ij)
            
            # 4. P_ij no normalizado
            P_ij = torch.exp(S_ij - m_new.unsqueeze(-1))  # (Br, Bc)
            
            # 5. Correccion de la suma previa y adicion del nuevo bloque
            alpha = torch.exp(m_block - m_new)  # Factor de re-escalado
            l_new = l_block * alpha + P_ij.sum(dim=-1)
            
            # 6. Actualizacion recursiva de la salida acumulada O
            # O_new = diag(alpha) * O + P_ij @ V_j
            o_block = (o_block * alpha.unsqueeze(-1)) + (P_ij @ v_block)
            
            m_block = m_new
            l_block = l_new
            
        # Normalizar el bloque de salida con su denominador final
        O[i:i+block_size_q] = o_block / l_block.unsqueeze(-1)
        m[i:i+block_size_q] = m_block
        l[i:i+block_size_q] = l_block
        
    return O

print("✅ Función flash_attention_forward compilada exitosamente")

### Comparación Numérica Rigurosa: Atención Estándar vs FlashAttention
Verifiquemos que FlashAttention produce exactamente las mismas salidas que la atención tradicional de PyTorch:

In [ ]:
# Parametros de prueba
seq_len = 128
d_head = 64

Q = torch.randn(seq_len, d_head)
K = torch.randn(seq_len, d_head)
V = torch.randn(seq_len, d_head)

# 1. Atencion Estandar (materializando matriz N x N)
scale = 1.0 / math.sqrt(d_head)
scores = (Q @ K.t()) * scale
attn_weights = F.softmax(scores, dim=-1)
out_standard = attn_weights @ V

# 2. FlashAttention (por bloques de 32x32, sin matriz N x N)
out_flash = flash_attention_forward(Q, K, V, block_size_q=32, block_size_kv=32)

# Comparacion de discrepancia maxima
max_diff = torch.max(torch.abs(out_standard - out_flash)).item()
print(f"Discrepancia Máxima entre Atención Estándar y FlashAttention: {max_diff:.2e}")
assert torch.allclose(out_standard, out_flash, atol=1e-5)
print("🚀 ¡Identidad numérica exacta demostrada! FlashAttention calcula la atención completa sin almacenar la matriz cuadrática N x N")

### Análisis Gráfico: La Reducción Exponencial de Memoria
Visualicemos la huella de memoria requerida para la matriz de atención a medida que la longitud de contexto crece:

In [ ]:
seq_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768])
num_heads = 32
bytes_per_elem = 2.0  # FP16

# Memoria de atencion estandar: num_heads * N^2 * bytes
mem_standard_gb = (num_heads * (seq_lengths ** 2) * bytes_per_elem) / 1e9

# Memoria de FlashAttention (solo bloques en SRAM + buffers O(N))
block_size = 64
mem_flash_gb = (num_heads * seq_lengths * 64 * bytes_per_elem) / 1e9  # Escala O(N)

plt.figure(figsize=(9, 5))
plt.plot(seq_lengths, mem_standard_gb, 'r-o', linewidth=2.5, label='Atención Estándar $O(N^2)$')
plt.plot(seq_lengths, mem_flash_gb, 'g-s', linewidth=2.5, label='FlashAttention $O(N)$')

plt.axhline(80.0, color='gray', linestyle='--', label='Capacidad GPU A100 (80 GB)')
plt.axhline(16.0, color='purple', linestyle=':', label='Capacidad GPU Consumo (16 GB)')

plt.title('Huella de Memoria de la Matriz de Atención según Contexto (32 cabezas)', fontsize=12, fontweight='bold')
plt.xlabel('Longitud de Secuencia ($N$ tokens)', fontsize=11)
plt.ylabel('Memoria Requerida (GB)', fontsize=11)
plt.yscale('log')
plt.grid(True, which="both", ls=":", alpha=0.5)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()
print(f"Para N=32.768 tokens:")
print(f"- Atención estándar necesitaría: {mem_standard_gb[-1]:.1f} GB (Inviable en cualquier GPU)")
print(f"- FlashAttention requiere:        {mem_flash_gb[-1]:.3f} GB (¡Apenas {mem_flash_gb[-1]*1024:.0f} MB!)")

---

## 4. ⚡ Transición a PyTorch Moderno

En PyTorch 2.0+, la función nativa `torch.nn.functional.scaled_dot_product_attention` (SDPA) ejecuta FlashAttention automáticamente en hardware compatible:

```python
# En PyTorch 2.0+ con GPU NVIDIA/AMD compatible:
with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
    # Computo ultra-optimizado a nivel de registros/SRAM con FlashAttention-2 nativo en C++
    out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
```

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Verificación de la Complejidad de Accesos a Memoria (IO-Complexity)
Tri Dao demostró que mientras la atención estándar requiere $O(N d + N^2)$ accesos a HBM, FlashAttention solo requiere $O(N^2 d^2 / M)$ accesos donde $M$ es la memoria SRAM del procesador. Comprobemos analíticamente el ratio de reducción de accesos:

In [ ]:
def calculate_hbm_reads(N: int, d: int, sram_bytes: int) -> Tuple[float, float]:
    bytes_per_elem = 2.0  # FP16
    # Estandar: Lee Q, K, V (3*N*d), escribe y lee S (2*N^2), escribe y lee P (2*N^2), escribe O (N*d)
    standard_bytes = (4 * (N ** 2) + 4 * N * d) * bytes_per_elem
    
    # FlashAttention: Divide en bloques que caben en SRAM M
    # Accesos a HBM = (N^2 * d^2 / M) * constante
    flash_bytes = (4 * N * d + (N ** 2 * d * bytes_per_elem) / (sram_bytes / 4)) * bytes_per_elem
    return standard_bytes, flash_bytes

std_b, flash_b = calculate_hbm_reads(N=4096, d=64, sram_bytes=100_000)  # 100 KB SRAM
print(f"Accesos HBM Atención Estándar (N=4096): {std_b / 1e6:.1f} MB")
print(f"Accesos HBM FlashAttention    (N=4096): {flash_b / 1e6:.1f} MB")
print(f"Factor de reducción de tráfico en bus:  {std_b / flash_b:.1f}x")
assert std_b > flash_b

### Reto 2 (Para resolver): Implementar FlashAttention con Máscara Causal y Poda de Bloques
En atención autorregresiva (decodificador GPT), las posiciones $j > i$ están enmascaradas con $-\infty$. En FlashAttention, si el bloque de claves $K_j$ está completamente en el futuro con respecto al bloque de consultas $Q_i$ ($j \cdot B_c > (i+1) \cdot B_r$), **¡podemos ignorar el bloque entero sin computar nada!**

Implementa a continuación la función `causal_flash_attention(...)` con salto de bloques superiores:

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def causal_flash_attention(
    Q: torch.Tensor, 
    K: torch.Tensor, 
    V: torch.Tensor, 
    block_size_q: int = 16, 
    block_size_kv: int = 16
) -> torch.Tensor:
    """
    Implementa FlashAttention causal ahorrando el 50% de bloques del triangulo superior.
    """
    # Tu implementación aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Dao, T., et al. (2022):** *"FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"*, NeurIPS 2022. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)
   * *¿Qué leer?* Sección 3 ("Algorithm: FlashAttention"): la formulación del algoritmo por bloques y la prueba de complejidad IO.
2. **Dao, T. (2023):** *"FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning"*, ICLR 2024. [arXiv:2307.08691](https://arxiv.org/abs/2307.08691)
   * *¿Qué leer?* La inversión del orden de los bucles externos/internos para maximizar la saturación de los multiprocesadores.
3. **Milakov, M., & Gimelshtein, N. (2018):** *"Online normalizer calculation for softmax"*, NVIDIA. [arXiv:1805.02867](https://arxiv.org/abs/1805.02867)
   * *¿Qué leer?* La deducción analítica de la actualización recursiva del Softmax en una sola pasada.